In [1]:
%matplotlib inline
import pandas as pd
import glob
import matplotlib.pyplot as plt
import os
import numpy as np
import random
import scipy.stats as st
from collections import defaultdict
import seaborn as sns
import csv
import math
from scipy.stats import ttest_1samp
from scipy.stats import chi2
from scipy.spatial.distance import mahalanobis
import copy
from scipy.interpolate import PchipInterpolator
from scipy.interpolate import interp1d
from scipy.signal import savgol_filter
from mpl_toolkits import mplot3d
import matplotlib.cm as cm
from matplotlib.collections import LineCollection
from matplotlib.colors import LogNorm
from scipy.stats import zscore

In [2]:
sf_sequence_105 = np.array([0,3,6,2,4,0,0,1,5,2,6,5,3,3,4,4,0,4,3,6,1,2,5,0,6,1,2,6,4,2,4,4,5,2,4,5,6,6,3,5,6,5,
                           2,1,2,2,4,1,5,4,0,1,4,1,2,3,5,2,0,4,4,2,3,0,6,4,1,2,5,0,6,0,3,6,1,3,5,0,6,0,1,6,3,5])
sf_sequence_70 = np.array([0, 1, 4, 5, 1, 0, 6, 0, 5, 3, 2, 1, 3, 6, 2, 4, 6, 4, 6, 0, 2, 3, 6, 1, 4, 6, 2, 3, 2, 1, 1, 3, 1, 5, 5, 0, 
                           5, 4, 5, 2, 4, 3, 2, 5, 2, 2, 4, 1, 6, 1, 0, 4, 5, 4, 6, 3, 3, 0, 0, 0, 5, 6, 5, 3, 6, 0, 2, 1, 4, 3])


In [ ]:
def align_pupil_to_neural_global(pupil_trace, neural_n_frames):
    """
    Align entire pupil trace to match neural recording length
    Centers the pupil trace on the neural trace and stretches/compresses
    both ends proportionally
    
    Parameters:
    -----------
    pupil_trace : array
        Cleaned pupil trace (with interpolation)
    neural_n_frames : int
        Total number of frames in neural recording
    
    Returns:
    --------
    aligned_pupil : array of length neural_n_frames
    """
    pupil_trace = np.array(pupil_trace)
    if pupil_trace.ndim > 1:
        pupil_trace = pupil_trace.flatten()
    N_pupil = len(pupil_trace)
    N_neural = neural_n_frames
    
    # Calculate expected pupil frames based on neural duration
    neural_duration = (N_neural / 30) 
    expected_pupil_frames = int(neural_duration * 25) 
    
    print(f"\n{'='*60}")
    print("GLOBAL ALIGNMENT (CENTER-BASED)")
    print(f"{'='*60}")
    print(f"Neural: {N_neural} frames at 30 Hz "
          f"= {neural_duration:.2f}s")
    print(f"Pupil:  {N_pupil} frames at 25 Hz "
          f"= {N_pupil/25:.2f}s")
    print(f"Expected pupil frames: {expected_pupil_frames}")
    
    # Calculate difference
    frame_diff = N_pupil - expected_pupil_frames
    time_diff = frame_diff / 25
    
    print(f"Difference: {frame_diff} frames ({time_diff:.2f}s)")

    # METHOD 1: Simple linear stretching (recommended)
    # Maps pupil timepoints to neural timepoints linearly
    # This naturally centers the alignment
    print(f"\nAlignment method: Linear stretch/compress")
    print(f"  Pupil midpoint: frame {N_pupil//2} → Neural midpoint: frame {N_neural//2}")
    
    # Create normalized time axes (0 to 1)
    # Both start at 0 and end at 1, so midpoints naturally align
    pupil_time_norm = np.linspace(0, 1, N_pupil)
    neural_time_norm = np.linspace(0, 1, N_neural)
    
    # Interpolate pupil to match neural length
    print(f"pupil_time_norm type: {type(pupil_time_norm)}")
    print(f"pupil_time_norm shape: {pupil_time_norm.shape if hasattr(pupil_time_norm, 'shape') else 'no shape'}")
    print(f"pupil_time_norm length: {len(pupil_time_norm)}")

    print(f"pupil_trace type: {type(pupil_trace)}")
    print(f"pupil_trace shape: {pupil_trace.shape if hasattr(pupil_trace, 'shape') else 'no shape'}")
    print(f"pupil_trace length: {len(pupil_trace)}")

    # Check for NaN or inf
    print(f"pupil_time_norm has NaN: {np.any(np.isnan(pupil_time_norm))}")
    print(f"pupil_trace has NaN: {np.any(np.isnan(pupil_trace))}")
    interp_func = interp1d(pupil_time_norm, pupil_trace, 
                           kind='linear',
                           fill_value='extrapolate')
    aligned_pupil = interp_func(neural_time_norm)
    
    print(f"✓ Aligned pupil trace: {len(aligned_pupil)} frames")
    print(f"  Matches neural: {len(aligned_pupil) == N_neural}")
    
    # Verify centering
    pupil_mid_val = pupil_trace[N_pupil//2]
    aligned_mid_val = aligned_pupil[N_neural//2]
    print(f"  Pupil midpoint value: {pupil_mid_val:.2f}")
    print(f"  Aligned midpoint value: {aligned_mid_val:.2f}")
    print(f"  Difference: {abs(pupil_mid_val - aligned_mid_val):.2f}")

    return aligned_pupil

In [ ]:
def calculate_stimulus_onsets_from_frames(ops, neural_fps=30):
    """
    Calculate stimulus onsets using actual frame counts per trial
    
    Parameters:
    -----------
    ops : dict
        Suite2P ops dictionary containing 'frames_per_file'
    neural_fps : int
        Neural frame rate
    
    Returns:
    --------
    onset_frames : array
        Frame indices for each stimulus onset
    onset_times : array
        Time (seconds) for each stimulus onset
    frames_per_trial : array
        Actual number of frames in each trial
    """
    
    frames_per_file = ops['frames_per_file']
    
    onset_frames = np.concatenate([[0], np.cumsum(frames_per_file[:-1])])
    onset_times = onset_frames / neural_fps
    
    n_trials = len(onset_frames)
    
    print(f"\nCalculated {n_trials} stimulus onsets")
    print(f"First onset: frame {onset_frames[0]} ({onset_times[0]:.2f}s)")
    print(f"Second onset: frame {onset_frames[1]} ({onset_times[1]:.2f}s)")
    print(f"Last onset: frame {onset_frames[-1]} ({onset_times[-1]:.2f}s)")
    
    # Expected vs actual recording length
    total_frames_expected = np.sum(frames_per_file)
    total_frames_actual = ops['nframes']
    
    
    if total_frames_expected != total_frames_actual:
        print(f"  ⚠️ Mismatch: {total_frames_actual - total_frames_expected} frame difference")
    else:
        print(f"  ✓ Match!")
    
    
    return onset_frames, onset_times

In [ ]:
def estimate_stimulus_and_isi_frames(frames_per_file, 
                                     expected_trial_duration=6.3,
                                     expected_stim_duration=1.3,
                                     neural_fps=30):#change these parameters
    """
    Estimate how frames are split between stimulus and ISI for each trial
    
    Parameters:
    -----------
    frames_per_trial : array
        Actual frames per trial from ops['frames_per_file']
    expected_trial_duration : float
        Expected total trial duration (set these)
    expected_stim_duration : float
        Expected stimulus duration (set these)
    neural_fps : int
        Frame rate
    
    Returns:
    --------
    frames_stim_per_trial : array
        Estimated stimulus frames for each trial
    frames_isi_per_trial : array
        Estimated ISI frames for each trial
    """
    
    # Expected frames
    expected_total_frames = int(expected_trial_duration * neural_fps)  # 150
    expected_stim_frames = int(expected_stim_duration * neural_fps)     # 9
    expected_isi_frames = expected_total_frames - expected_stim_frames  # 141
    
    # Calculate proportion of each trial that's stimulus vs ISI
    stim_proportion = expected_stim_duration / expected_trial_duration  # 0.3 / 5.0 = 0.06
    
    # Allocate frames proportionally
    frames_stim_per_trial = np.round(frames_per_trial * stim_proportion).astype(int)
    frames_isi_per_trial = frames_per_trial - frames_stim_per_trial
    
    # Ensure stimulus is at least 1 frame
    frames_stim_per_trial = np.maximum(frames_stim_per_trial, 1)
    
    return frames_stim_per_trial, frames_isi_per_trial


In [ ]:
def extract_pupil_by_stimulus(aligned_pupil, onset_frames, 
                                     frames_per_trial, sf_sequence,
                                     neural_fps=30,
                                     stimulus_duration=1.3): #change these parameters

    """
    Extract pupil segments - SIMPLIFIED version
    Just use fixed stimulus duration, let ISI be whatever's left
    """
    
    # FIXED stimulus period
    frames_stimulus = int(stimulus_duration * neural_fps)  

    
    # Initialize storage
    pupil_stimulus = {}
    pupil_isi = {}
    pupil_baseline = {}
    pupil_full_trial = {}
    
    avg_stimulus = {}
    avg_isi = {}
    avg_baseline = {}
    peak_stimulus = {}
    
    n_trials = min(len(onset_frames), len(sf_sequence), len(frames_per_trial))
    
    for trial_idx in range(n_trials):
        
        stimulus = sf_sequence[trial_idx]
        onset = onset_frames[trial_idx]
        total_frames = frames_per_trial[trial_idx]
        
        # Check bounds
        if onset + total_frames > len(aligned_pupil):
            print(f"⚠️ Trial {trial_idx} extends beyond recording, stopping at trial {trial_idx-1}")
            break
        
        # ====================================================================
        # STIMULUS period
        # ====================================================================
        stim_end = onset + frames_stimulus
        pupil_stim = aligned_pupil[onset:stim_end]
        
        # ====================================================================
        # ISI AFTER stimulus: Remaining frames in this trial
        # ====================================================================
        pupil_isi_after = aligned_pupil[stim_end:onset + total_frames]
        
        # ====================================================================
        # BASELINE: ISI from previous trial
        # ====================================================================
        if trial_idx == 0:
            # First trial has no baseline - use NaN
            pupil_baseline_before = np.array([np.nan])
            avg_base = np.nan  # ← FIX: Set directly to NaN
        else:
            # Previous trial's ISI period
            prev_onset = onset_frames[trial_idx - 1]
            prev_stim_end = prev_onset + frames_stimulus
            
            # ISI goes from end of previous stimulus to start of current trial
            pupil_baseline_before = aligned_pupil[prev_stim_end:onset]
            
            # Calculate baseline average (should have data)
            if len(pupil_baseline_before) > 0:
                avg_base = np.mean(pupil_baseline_before)
            else:
                avg_base = np.nan  # Edge case: no frames between trials
        
        # ====================================================================
        # FULL TRIAL
        # ====================================================================
        pupil_full = aligned_pupil[onset:onset + total_frames]
        
        # Calculate statistics
        avg_stim = np.mean(pupil_stim)
        avg_isi_after = np.mean(pupil_isi_after)
        # avg_base already calculated above
        peak_stim = np.max(pupil_stim)
        
        # Store by stimulus type
        if stimulus not in pupil_stimulus:
            pupil_stimulus[stimulus] = []
            pupil_isi[stimulus] = []
            pupil_baseline[stimulus] = []
            pupil_full_trial[stimulus] = []
            avg_stimulus[stimulus] = []
            avg_isi[stimulus] = []
            avg_baseline[stimulus] = []
            peak_stimulus[stimulus] = []
        
        pupil_stimulus[stimulus].append(pupil_stim)
        pupil_isi[stimulus].append(pupil_isi_after)
        pupil_baseline[stimulus].append(pupil_baseline_before)
        pupil_full_trial[stimulus].append(pupil_full)
        avg_stimulus[stimulus].append(avg_stim)
        avg_isi[stimulus].append(avg_isi_after)
        avg_baseline[stimulus].append(avg_base)
        peak_stimulus[stimulus].append(peak_stim)
    
    
    for stim in sorted(pupil_stimulus.keys()):
        # Average values can be arrays (scalar per trial)
        avg_stimulus[stim] = np.array(avg_stimulus[stim])
        avg_isi[stim] = np.array(avg_isi[stim])
        avg_baseline[stim] = np.array(avg_baseline[stim])
        peak_stimulus[stim] = np.array(peak_stimulus[stim])
        
        # Traces stay as lists (variable length for ISI/baseline)
        # But stimulus is fixed length, so we can convert it
        pupil_stimulus[stim] = np.array(pupil_stimulus[stim])  # Shape: (n_trials, 9)
        
        stim_name = "Background" if stim == 0 else f"Stimulus {stim}"
        n_trials_stim = len(pupil_stimulus[stim])
        
        # Count how many trials have valid baseline
        n_with_baseline = np.sum(~np.isnan(avg_baseline[stim]))


    
    results = {
        'pupil_stimulus': pupil_stimulus,
        'pupil_isi': pupil_isi,
        'pupil_baseline': pupil_baseline,
        'pupil_full_trial': pupil_full_trial,
        'avg_stimulus': avg_stimulus,
        'avg_isi': avg_isi,
        'avg_baseline': avg_baseline,
        'peak_stimulus': peak_stimulus,
        'onset_frames': onset_frames,
        'frames_per_trial': frames_per_trial,
        'frames_stimulus': frames_stimulus
    }
    
    return results

In [ ]:
def calculate_stimulus_onsets_from_frames_robust(ops, neural_fps=30, 
                                          min_trial_frames=100):
    """
    Calculate stimulus onsets using actual frame counts per trial
    Merges anomalously short trials (1-2 frames) with the following trial
    
    Parameters:
    -----------
    ops : dict
        Suite2P ops dictionary containing 'frames_per_file'
    neural_fps : int
        Neural frame rate
    min_trial_frames : int
        Minimum frames for a valid trial (default: 100)
        Trials with fewer frames are merged with the next trial
    
    Returns:
    --------
    onset_frames : array
        Frame indices for each stimulus onset
    onset_times : array
        Time (seconds) for each stimulus onset
    frames_per_trial : array
        Actual number of frames in each trial (after merging)
    """
    
    frames_per_file = ops['frames_per_file'].copy()
    
    # ========================================================================
    # STEP 1: Identify and merge short trials
    # ========================================================================
    
    short_trials = np.where(frames_per_file < min_trial_frames)[0]
    
    if len(short_trials) > 0:
        print(f"\n⚠️  Found {len(short_trials)} anomalously short trials:")
        for idx in short_trials:
            print(f"  Trial {idx}: {frames_per_file[idx]} frames")
        
        print(f"\nMerging short trials with following trial...")
        
        # Create merged frames array
        frames_merged = []
        skip_next = False
        
        for i in range(len(frames_per_file)):
            
            if skip_next:
                skip_next = False
                continue
            
            current_frames = frames_per_file[i]
            
            # Check if this trial is too short
            if current_frames < min_trial_frames:
                
                # Merge with next trial
                if i + 1 < len(frames_per_file):
                    merged_frames = current_frames + frames_per_file[i + 1]
                    frames_merged.append(merged_frames)
                    skip_next = True
                    
                    print(f"  Merged trial {i} ({current_frames} frames) + "
                          f"trial {i+1} ({frames_per_file[i+1]} frames) "
                          f"= {merged_frames} frames")
                else:
                    # Last trial is short - just keep it
                    frames_merged.append(current_frames)
                    print(f"  ⚠️  Trial {i} is last trial and short - keeping as is")
            else:
                # Normal trial - keep as is
                frames_merged.append(current_frames)
        
        frames_per_file = np.array(frames_merged)
        
        print(f"\nAfter merging:")
        print(f"  Total trials: {len(frames_per_file)} (was {len(ops['frames_per_file'])})")
        print(f"  Frames per trial - Min: {np.min(frames_per_file)}, "
              f"Max: {np.max(frames_per_file)}, Mean: {np.mean(frames_per_file):.2f}")
    else:
        print("\n✓ No anomalously short trials found")
    
    # ========================================================================
    # STEP 2: Calculate onsets from merged trials
    # ========================================================================
    
    onset_frames = np.concatenate([[0], np.cumsum(frames_per_file[:-1])])
    onset_times = onset_frames / neural_fps
    
    n_trials = len(onset_frames)
    
    print(f"\nCalculated {n_trials} stimulus onsets")
    print(f"First 5 onsets:")
    for i in range(min(5, n_trials)):
        print(f"  Trial {i}: frame {onset_frames[i]:5d} ({onset_times[i]:7.2f}s), "
              f"{frames_per_file[i]} frames")
    
    print(f"Last onset: frame {onset_frames[-1]} ({onset_times[-1]:.2f}s)")
    
    return onset_frames, onset_times

In [ ]:
def pull_pupil(all_sheets_dict, mouse):
    """
    Extract pupil data for a specific mouse
    
    Handles columns like: 'M05r10_pupil_size'
    """
    pupil_trace_dict = {}
    
    for sheet_name, df in all_sheets_dict.items():
        # Check if this sheet belongs to this mouse
        if not sheet_name.startswith(mouse):
            continue
        
        pupil_trace_dict[sheet_name] = {}
        
        for column_name in df.columns:
            # Only process columns that start with mouse ID
            if not column_name.startswith(mouse):
                continue
            
            # Extract recording number from column name
            # Example: 'M05r10_pupil_size' → 'M05r10'
            try:
                # Split by underscore and take first part
                recording_num = column_name.split('_')[0]
                
                # Verify it looks like 'M05r10'
                if not recording_num.startswith(mouse + 'r'):
                    continue
                
                # Create or append to recording's DataFrame
                if recording_num not in pupil_trace_dict[sheet_name]:
                    pupil_trace_dict[sheet_name][recording_num] = pd.DataFrame()
                
                pupil_trace_dict[sheet_name][recording_num][column_name] = df[column_name]
                
            except Exception as e:
                print(f"[WARN] Could not process column '{column_name}': {e}")
                continue
    
    # Drop NaN rows
    for sheet_name, recordings in pupil_trace_dict.items():
        for recording_num, df in recordings.items():
            recordings[recording_num] = df.dropna()
    
    return pupil_trace_dict

In [ ]:
def entire_pupil_analysis(neural_data_path, pupil_data, sf_sequence_70, sf_sequence_105):
    ops = np.load(neural_data_path, allow_pickle=True).item() #paste the file path in ops
    neural_n_frames = ops['nframes']
    pupil_trace = pupil_data #redo this according to file path
    frames_per_trial = ops['frames_per_file']
    
    aligned_pupil = align_pupil_to_neural_global(pupil_trace, neural_n_frames)
    onset_frames, onset_times = calculate_stimulus_onsets_from_frames_robust(ops)
    
    # =========================================================
    # NEW: Trim incomplete trials at the end
    # =========================================================
    expected_frames = np.median(frames_per_trial)
    min_valid_frames = 0.8 * expected_frames  # tolerate some jitter

    valid_trial_mask = frames_per_trial >= min_valid_frames

    # Keep only *leading* valid trials (stop at first bad one)
    cutoff_idx = len(frames_per_trial)
    for i, valid in enumerate(valid_trial_mask):
        if not valid:
            cutoff_idx = i
            break

    frames_per_trial = frames_per_trial[:cutoff_idx]
    onset_frames = onset_frames[:cutoff_idx]

    # =========================================================

    if len(onset_frames) == 105:
        sf_sequence = sf_sequence_105
    elif len(onset_frames) == 70:
        sf_sequence = sf_sequence_70
    elif len(onset_frames) < len(sf_sequence_70):
        diff = len(sf_sequence_70) - len(onset_frames)
        sf_sequence = sf_sequence_70.copy()[:-diff]

    extracted_pupil_dict = extract_pupil_by_stimulus(aligned_pupil, onset_frames, 
                                     frames_per_trial, sf_sequence,
                                     neural_fps=30,
                                     stimulus_duration=1.3)

    average_pupil_trace = {}
    average_pupil_increase = {}
    

    for x in extracted_pupil_dict['pupil_stimulus']:
        index = 0
        average_pupil_trace[x] = []
        average_pupil_increase[x] = []
        for values in extracted_pupil_dict['pupil_stimulus'][x]:
            average_pupil_trace[x].append(np.average(values))
            baseline_value = np.average(extracted_pupil_dict['pupil_baseline'][x][index][-30:])
            average_pupil_increase[x].append(np.average([(y-baseline_value) for y in values]))
            index += 1

    return average_pupil_trace, average_pupil_increase

In [ ]:
def entire_pupil_values(neural_data_path, pupil_data, sf_sequence_70, sf_sequence_105):
    ops = np.load(neural_data_path, allow_pickle=True).item() #paste the file path in ops
    neural_n_frames = ops['nframes']
    print(neural_n_frames)
    pupil_trace = pupil_data #redo this according to file path
    frames_per_trial = ops['frames_per_file']
    print(frames_per_trial)
    
    aligned_pupil = align_pupil_to_neural_global(pupil_trace, neural_n_frames)
    onset_frames, onset_times = calculate_stimulus_onsets_from_frames_robust(ops)

        # =========================================================
    # NEW: Trim incomplete trials at the end
    # =========================================================
    expected_frames = np.median(frames_per_trial)
    min_valid_frames = 0.8 * expected_frames

    valid_trial_mask = frames_per_trial >= min_valid_frames

    cutoff_idx = len(frames_per_trial)
    for i, valid in enumerate(valid_trial_mask):
        if not valid:
            cutoff_idx = i
            break

    frames_per_trial = frames_per_trial[:cutoff_idx]
    onset_frames = onset_frames[:cutoff_idx]
    # =========================================================

    if len(onset_frames) == 105:
        sf_sequence = sf_sequence_105
    elif len(onset_frames) == 70:
        sf_sequence = sf_sequence_70
    elif len(onset_frames) < len(sf_sequence_70):
        diff = len(sf_sequence_70) - len(onset_frames)
        sf_sequence = sf_sequence_70.copy()[:-diff]

    extracted_pupil_dict = extract_pupil_by_stimulus(aligned_pupil, onset_frames, 
                                     frames_per_trial, sf_sequence,
                                     neural_fps=30,
                                     stimulus_duration=1.3)
    return extracted_pupil_dict

In [ ]:
def load_block_contrast_analysis(block_id, pupil_trace_dict,
                        sf_sequence_70, sf_sequence_105,
                        verbose=True):
    """
    Load contrast data where:
    - Pupil Excel uses continuous numbering (r1-r16)
    - Ops files use per-day numbering (r1-r4 each day)
    
    Parameters:
    -----------
    block_id : str
        Mouse ID (e.g., 'M05')
    pupil_trace_dict : dict
        Pupil data from Excel file
    """
    
    out = {f'd{i}': {} for i in range(1, 5)}  # d1, d2, d3, d4
    
    # Iterate through continuous run numbers (1-16)
    for continuous_run in range(1, 17):
        
        # Determine which day and ops run this corresponds to
        day = get_day_from_continuous(continuous_run)
        ops_run = get_ops_run_from_continuous(continuous_run)
        
        # Keys for accessing data
        day_key = f'{block_id}d{day}'
        pupil_run_key = f'{block_id}r{continuous_run}'  # e.g., 'M05r5'
        
        # Ops file path uses day and ops_run (1-4)
        ops_path = f'Contrast_2P_Python/{block_id}d{day}r{ops_run}/ops.npy'
        
        if verbose and ops_run == 1:  # Print once per day
            print(f'\nDay {day} mapping:')
            print(f'  Pupil runs: r{(day-1)*4+1}-r{day*4} → Ops runs: r1-r4')
        
        # Check ops file
        if not os.path.exists(ops_path):
            if verbose:
                print(f'[WARN] Missing ops: {ops_path} (pupil r{continuous_run})')
            out[f'd{day}'][f'r{ops_run}'] = None
            continue
        
        # Check pupil data
        try:
            pupil_data = pupil_trace_dict[day_key][pupil_run_key][f'{pupil_run_key}_pupil_size']
        except KeyError:
            if verbose:
                print(f'[WARN] Missing pupil: {day_key} / {pupil_run_key}')
            out[f'd{day}'][f'r{ops_run}'] = None
            continue
        
        # Run analysis
        try:
            result = entire_pupil_analysis( ###change here
                ops_path,
                pupil_data,
                sf_sequence_70,
                sf_sequence_105
            )
            out[f'd{day}'][f'r{ops_run}'] = result
            
            if verbose:
                print(f'  ✓ Processed: pupil r{continuous_run} → '
                      f'ops d{day}r{ops_run}')
                
        except Exception as e:
            if verbose:
                print(f'[ERROR] Failed pupil r{continuous_run} '
                      f'(ops d{day}r{ops_run}): {e}')
            out[f'd{day}'][f'r{ops_run}'] = None
    
    return out

In [ ]:
def load_block_contrast_values(block_id, pupil_trace_dict,
                        sf_sequence_70, sf_sequence_105,
                        verbose=True):
    """
    Load contrast data where:
    - Pupil Excel uses continuous numbering (r1-r16)
    - Ops files use per-day numbering (r1-r4 each day)
    
    Parameters:
    -----------
    block_id : str
        Mouse ID (e.g., 'M05')
    pupil_trace_dict : dict
        Pupil data from Excel file
    """
    
    out = {f'd{i}': {} for i in range(1, 5)}  # d1, d2, d3, d4
    
    # Iterate through continuous run numbers (1-16)
    for continuous_run in range(1, 17):
        
        # Determine which day and ops run this corresponds to
        day = get_day_from_continuous(continuous_run)
        ops_run = get_ops_run_from_continuous(continuous_run)
        
        # Keys for accessing data
        day_key = f'{block_id}d{day}'
        pupil_run_key = f'{block_id}r{continuous_run}'  # e.g., 'M05r5'
        
        # Ops file path uses day and ops_run (1-4)
        ops_path = f'Contrast_2P_Python/{block_id}d{day}r{ops_run}/ops.npy'
        
        if verbose and ops_run == 1:  # Print once per day
            print(f'\nDay {day} mapping:')
            print(f'  Pupil runs: r{(day-1)*4+1}-r{day*4} → Ops runs: r1-r4')
        
        # Check ops file
        if not os.path.exists(ops_path):
            if verbose:
                print(f'[WARN] Missing ops: {ops_path} (pupil r{continuous_run})')
            out[f'd{day}'][f'r{ops_run}'] = None
            continue
        
        # Check pupil data
        try:
            pupil_data = pupil_trace_dict[day_key][pupil_run_key][f'{pupil_run_key}_pupil_size']
        except KeyError:
            if verbose:
                print(f'[WARN] Missing pupil: {day_key} / {pupil_run_key}')
            out[f'd{day}'][f'r{ops_run}'] = None
            continue
        
        # Run analysis
        try:
            result = entire_pupil_values( ###change here
                ops_path,
                pupil_data,
                sf_sequence_70,
                sf_sequence_105
            )
            out[f'd{day}'][f'r{ops_run}'] = result
            
            if verbose:
                print(f'  ✓ Processed: pupil r{continuous_run} → '
                      f'ops d{day}r{ops_run}')
                
        except Exception as e:
            if verbose:
                print(f'[ERROR] Failed pupil r{continuous_run} '
                      f'(ops d{day}r{ops_run}): {e}')
            out[f'd{day}'][f'r{ops_run}'] = None
    
    return out

In [ ]:
def get_ops_run_from_continuous(continuous_run):
    """
    Convert continuous run number (1-16) to within-day run number (1-4)
    
    Examples:
    - r1 → r1 (day 1, run 1)
    - r5 → r1 (day 2, run 1) 
    - r9 → r1 (day 3, run 1)
    - r13 → r1 (day 4, run 1)
    - r6 → r2 (day 2, run 2)
    - r14 → r2 (day 4, run 2)
    """
    ops_run = ((continuous_run - 1) % 4) + 1
    return ops_run


def get_day_from_continuous(continuous_run):
    """
    Convert continuous run number (1-16) to day number (1-4)
    
    Examples:
    - r1-r4 → day 1
    - r5-r8 → day 2
    - r9-r12 → day 3
    - r13-r16 → day 4
    """
    day = ((continuous_run - 1) // 4) + 1
    return day


# Test the functions
print("Testing mapping:")
for continuous in [1, 2, 5, 6, 9, 10, 13, 14, 16]:
    day = get_day_from_continuous(continuous)
    ops_run = get_ops_run_from_continuous(continuous)
    print(f"r{continuous:2d} → Day {day}, ops r{ops_run}")

In [ ]:
file_path = 'contrast_pupil_size_data.xlsx' #change file path here
all_sheets_dict = pd.read_excel(file_path, sheet_name=None)

In [ ]:
#get maximum pupil for each day
Absolute_Maximum_Pupil = {}

for days in all_sheets_dict:
    mouse = days[0:3]
    day = days[-2:]

    if mouse not in Absolute_Maximum_Pupil:
        Absolute_Maximum_Pupil[mouse] = {}

    temp_dict = {}

    for column in all_sheets_dict[days]:
        if 'pupil_size' in column:
            values = all_sheets_dict[days][column]

            if len(values) > 0:
                temp_dict[column] = np.nanmax(values)

    if temp_dict:
        Absolute_Maximum_Pupil[mouse][day] = np.nanmax(list(temp_dict.values()))
    else:
        Absolute_Maximum_Pupil[mouse][day] = None

In [ ]:
def export_absolute_max_to_excel(data_dict, output_file="absolute_max_contrast_pupil.xlsx"):
    """
    Writes the nested mouse→day dictionary to Excel.

    Rows = mice
    Columns = days (d1–d4)
    """

    # -----------------------------------
    # Convert to DataFrame directly
    # -----------------------------------
    df = pd.DataFrame.from_dict(data_dict, orient="index")
    desired_order = ["d1", "d2", "d3", "d4"]
    df = df.reindex(columns=desired_order)
    df = df.sort_index()

    # -----------------------------------
    # Save to Excel
    # -----------------------------------
    df.to_excel(output_file, engine="openpyxl")

    print(f"Saved to: {output_file}")

In [ ]:
export_absolute_max_to_excel(Absolute_Maximum_Pupil, output_file="absolute_max_contrast_pupil.xlsx")

In [ ]:

mice = ['M05', 'M07', 'M10', 'M12', 'M14', 'M15', 'M21']

complete_dict = {}

for mouse in mice:
    print(f"\n{'='*70}")
    print(f"Processing {mouse}")
    print(f"{'='*70}")
    
    pupil_trace_dict = pull_pupil(all_sheets_dict, mouse)
    
    complete_analysis_dict[mouse] = load_block_contrast_analysis(
        block_id=mouse,
        pupil_trace_dict=pupil_trace_dict,
        sf_sequence_70=sf_sequence_70,
        sf_sequence_105=sf_sequence_105,
        verbose=True
    )

    complete_values_dict[mouse] = load_block_contrast_values(
        block_id=mouse,
        pupil_trace_dict=pupil_trace_dict,
        sf_sequence_70=sf_sequence_70,
        sf_sequence_105=sf_sequence_105,
        verbose=True
    )

print(f"\n{'='*70}")
print("PROCESSING COMPLETE")
print(f"{'='*70}")

In [ ]:
sf_index_to_value = {
    0: 0.0,
    1: 0.1,
    2: 0.3,
    3: 0.5,
    4: 0.7,
    5: 0.9,
    6: 1.1, 
}


def export_mouse_timecourses(
    complete_dict,
    sf_index_to_value,
    output_folder="pupil_excels",
    baseline_frames=20,
):
    """
    Creates ONE Excel file per mouse.
    Each sheet = one spatial frequency.
    Rows are in long format with baseline → stimulus → ISI.

    Fixes:
    - ensures at least one visible sheet
    - avoids writer scope errors
    - handles missing keys safely
    """

    os.makedirs(output_folder, exist_ok=True)

    for mouse, mouse_dict in complete_dict.items():

        output_path = os.path.join(
            output_folder,
            f"{mouse}_pupil_timecourses.xlsx"
        )

        wrote_any_sheet = False  # ⭐ critical fix

        with pd.ExcelWriter(output_path, engine="openpyxl") as writer:

            # -------------------------------------------------
            # Collect spatial frequencies safely
            # -------------------------------------------------
            all_sfs = set()

            for day_dict in mouse_dict.values():
                for run_dict in day_dict.values():
                    if not isinstance(run_dict, dict):
                        continue

                    stim_dict = run_dict.get("pupil_stimulus", {})
                    all_sfs.update(stim_dict.keys())

            # -------------------------------------------------
            # ONE SHEET PER SPATIAL FREQUENCY
            # -------------------------------------------------
            for sf_idx in sorted(all_sfs):

                if sf_idx not in sf_index_to_value:
                    continue

                spatial_freq = sf_index_to_value[sf_idx]
                rows = []

                # =============================================
                # LOOP THROUGH DATA
                # =============================================
                for day, day_dict in mouse_dict.items():

                    phase = "pre" if day in ["d1", "d2"] else "post"

                    for run, run_dict in day_dict.items():

                        if not isinstance(run_dict, dict):
                            continue

                        region = (
                            "peripheral" if run in ["r1", "r3"]
                            else "central"
                        )

                        stim_trials = run_dict.get("pupil_stimulus", {}).get(sf_idx, [])
                        isi_trials  = run_dict.get("pupil_isi", {}).get(sf_idx, [])
                        base_trials = run_dict.get("pupil_baseline", {}).get(sf_idx, [])

                        n_trials = min(len(stim_trials),
                                       len(isi_trials),
                                       len(base_trials))

                        # =========================================
                        # LOOP THROUGH TRIALS
                        # =========================================
                        for trial_idx in range(n_trials):

                            stim_trace = stim_trials[trial_idx]
                            iti_trace  = isi_trials[trial_idx]
                            base_trace = base_trials[trial_idx]

                            if len(base_trace) == 0:
                                continue

                            base_tail = base_trace[-baseline_frames:]

                            # -----------------------------
                            # BASELINE
                            # -----------------------------
                            for frame_idx, val in enumerate(base_tail):
                                rows.append({
                                    "mouse": mouse,
                                    "phase": phase,
                                    "day": day,
                                    "run": run,
                                    "region": region,
                                    "trial": trial_idx + 1,
                                    "epoch": "baseline",
                                    "frame": frame_idx,
                                    "pupil": float(val),
                                })

                            # -----------------------------
                            # STIMULUS
                            # -----------------------------
                            for frame_idx, val in enumerate(stim_trace):
                                rows.append({
                                    "mouse": mouse,
                                    "phase": phase,
                                    "day": day,
                                    "run": run,
                                    "region": region,
                                    "trial": trial_idx + 1,
                                    "epoch": "stimulus",
                                    "frame": frame_idx,
                                    "pupil": float(val),
                                })

                            # -----------------------------
                            # ISI
                            # -----------------------------
                            for frame_idx, val in enumerate(iti_trace):
                                rows.append({
                                    "mouse": mouse,
                                    "phase": phase,
                                    "day": day,
                                    "run": run,
                                    "region": region,
                                    "trial": trial_idx + 1,
                                    "epoch": "ISI",
                                    "frame": frame_idx,
                                    "pupil": float(val),
                                })

                # =============================================
                # WRITE SHEET (only if data exists)
                # =============================================
                if rows:
                    df = pd.DataFrame(rows)

                    sheet_name = f"sf_{spatial_freq:.2f}".replace(".", "p")
                    df.to_excel(writer, sheet_name=sheet_name, index=False)

                    wrote_any_sheet = True  # ⭐ mark success

            # -------------------------------------------------
            # GUARANTEE at least one visible sheet
            # -------------------------------------------------
            if not wrote_any_sheet:
                pd.DataFrame({
                    "note": ["No valid data found for this mouse"]
                }).to_excel(writer, sheet_name="empty", index=False)

        print(f"Saved: {output_path}")

In [ ]:
def export_complete_dict_to_excel(
    complete_dict,
    sf_index_to_value,
    output_file="pupil_data.xlsx",
    average_or_stim_evoked
):
    """
    Exports complete_dict into a single Excel file.
    One sheet per mouse.
    Each row = one stimulus presentation value.
    """

    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

        for mouse, mouse_dict in complete_dict.items():

            rows = []

            for day, day_dict in mouse_dict.items():

                for rec, sf_tuple in day_dict.items():

                    if sf_tuple is None or not isinstance(sf_tuple, tuple):
                        continue

                    avg_dict = sf_tuple[average_or_stim_evoked]  # ONLY averages per SF sequence


                    for sf_idx, values in avg_dict.items():

                        if sf_idx not in sf_index_to_value:
                            continue

                        spatial_freq = sf_index_to_value[sf_idx]

                        if values is None:
                            continue

                        for val in values:
                            rows.append({
                                "day": day,
                                "recording": rec,
                                "sf_index": sf_idx,
                                "spatial_freq_cpd": spatial_freq,
                                "pupil_value": val
                            })

            # ---- convert to DataFrame ----
            df = pd.DataFrame(rows)

            # ---- write one sheet per mouse ----
            df.to_excel(writer, sheet_name=mouse, index=False)

    print(f"Excel file saved to: {output_file}")

In [ ]:
export_mouse_timecourses(
    complete_dict_values,
    sf_index_to_value,
    output_folder="contrast_pupil_excels",
    baseline_frames=20,   # ⭐ change to 30 if you prefer
)

In [ ]:
export_complete_dict_to_excel(
    complete_dict=complete_dict_analysis,
    sf_index_to_value=sf_index_to_value,
    output_file="average_pupil_contrast_data.xlsx",
    average_or_stim_evoked = 0
)

In [ ]:
export_complete_dict_to_excel(
    complete_dict=complete_dict_analysis,
    sf_index_to_value=sf_index_to_value,
    output_file="average_stimulus_evoked_contrast_data_M07.xlsx",
    average_or_stim_evoked = 1
)